# 21cmEMU v3: Full Emulator with 2D Power Spectrum

This tutorial demonstrates the complete capabilities of the v3 (MH) emulator, including:

1. **1D Summaries**: Global brightness temperature ($\overline{T}_b$), neutral fraction ($\overline{x}_{\mathrm{HI}}$), spin temperature ($T_s$), UV luminosity functions (UVLFs), and optical depth ($\tau_e$)
2. **2D Power Spectrum**: The 21-cm power spectrum $P(k_\perp, k_\parallel)$ emulated by a diffusion model
3. **Uncertainty Quantification**: Variance estimation from multiple diffusion model samples

The v3 emulator uses:
- An **LSTM-based encoder-decoder** architecture for 1D summaries
- A **score-based diffusion model** for the 2D power spectrum, enabling probabilistic sampling

**Requirements**: This tutorial assumes a GPU is available for 2D PS emulation.

In [ ]:
import numpy as np
import h5py
import matplotlib.pyplot as plt
from matplotlib import rcParams
import matplotlib.patches as mpatches

rcParams.update({'font.size': 14})

from py21cmemu import Emulator

## 1. Load Test Databases

We load the test databases containing simulation outputs to compare against emulator predictions.

In [ ]:
# Load the 1D summaries test set
test_1d_path = 'test_set.h5'

with h5py.File(test_1d_path, 'r') as f:
    print("Test set keys:", list(f.keys()))
    
    # Input parameters (11 astrophysical params)
    test_params = np.array(f['inputs'])
    
    # 1D summaries
    test_xHI = np.array(f['xHI'])[...,::-1] # flip redshift axis
    test_Tb = np.array(f['Tb'])[...,::-1] # flip redshift axis
    test_Ts = np.array(f['Ts_neutral'])[...,::-1] # flip redshift axis
    test_tau = np.array(f['tau_e'])
    test_PS_1D = np.array(f['PS_1D'])  # 1D PS for comparison
    
    # UVLFs: (N, 7 z-bins, 60 magnitudes)
    test_LFs_raw = np.array(f['LFs'])
    
    # Axes
    redshifts = np.array(f['redshifts'])[::-1] # flip redshift axis
    M_UV_all = np.array(f['M_UV'])
    LF_zs = np.array(f['UVLF_redshifts'])

# Filter valid samples (no NaN inputs)
valid_mask = ~np.isnan(test_params.mean(axis=1))
print(f"Valid samples: {valid_mask.sum()} / {len(test_params)}")

test_params = test_params[valid_mask]
test_xHI = test_xHI[valid_mask]
test_Tb = test_Tb[valid_mask]
test_Ts = test_Ts[valid_mask]
test_tau = test_tau[valid_mask]
test_LFs_raw = test_LFs_raw[valid_mask]

# Trim UVLFs to M_UV in [-20, -10]
M_UV = M_UV_all[10:-5]  # Crop to 45 magnitudes
test_LFs = test_LFs_raw[:, :, 10:-5]  # (N, 7, 45)

print("\nData shapes:")
print(f"  Params: {test_params.shape}")
print(f"  xHI: {test_xHI.shape}")
print(f"  Tb: {test_Tb.shape}")
print(f"  Ts: {test_Ts.shape}")
print(f"  tau: {test_tau.shape}")
print(f"  UVLFs: {test_LFs.shape}")
print(f"  Redshifts: {redshifts.shape}")

### Load 2D Power Spectrum Test Database

The 2D PS test database contains power spectra computed at multiple redshifts.

**Note**: If the 2D PS test database is not available, the PS comparison sections will use emulator properties for k-grids and generate predictions without ground truth comparison.

In [ ]:
import os

# Try to load the 2D PS test database
test_ps2d_path = 'ps_2d_test_subsample.h5'
HAS_PS2D_DB = os.path.exists(test_ps2d_path)

if HAS_PS2D_DB:
    with h5py.File(test_ps2d_path, 'r') as f:
        print("2D PS test set keys:", list(f.keys()))
        
        ps_params = np.array(f['input_params'])
        ps_redshifts_all = np.array(f['redshifts'])
        kperp = np.array(f['kperp'])
        kpar = np.array(f['kpar_64'])  # 32 kpar bins
        
        # 2D PS: seeds (individual realisations) and means (averaged over seeds)
        ps_2d_seeds = np.array(f['PS_2D_64_seeds'])
        ps_2d_means = np.array(f['PS_2D_64_means'])

    # Select a subset of redshifts for 2D PS emulation (to save time)
    n_z_subset = 10
    z_indices = np.linspace(0, len(ps_redshifts_all) - 1, n_z_subset, dtype=int)
    ps_redshifts = ps_redshifts_all[z_indices]
    
    # Also subset the 2D PS data to match
    ps_2d_means = ps_2d_means[:, z_indices]
    ps_2d_seeds = ps_2d_seeds[:, z_indices]

    print("\n2D PS data shapes:")
    print(f"  Params: {ps_params.shape}")
    print(f"  All redshifts: {ps_redshifts_all.shape}")
    print(f"  Selected redshifts ({n_z_subset}): {ps_redshifts.shape}")
    print(f"  Selected z values: {ps_redshifts}")
    print(f"  kperp: {kperp.shape}")
    print(f"  kpar: {kpar.shape}")
    print(f"  PS seeds (subsetted): {ps_2d_seeds.shape}")
    print(f"  PS means (subsetted): {ps_2d_means.shape}")
else:
    print("2D PS test database not found.")
    print("PS sections will use emulator properties and test params from 1D database.")
    print("\nTo download the 2D PS test database, see the 21cmEMU documentation.")
    ps_params = test_params  # Use same params as 1D test
    ps_redshifts = None  # Will use emulator defaults
    ps_2d_means = None
    ps_2d_seeds = None
    kperp = None  # Will use emulator properties  
    kpar = None   # Will use emulator properties

## 2. Initialize the v3 Emulator

We load the MH emulator with 2D PS emulation enabled.

In [ ]:
# Initialize the v3 (MH) emulator with 2D PS enabled
emu = Emulator(emulator='mh', emulate_2d_ps=True)

print(f"Emulator initialized: {emu.which_emulator}")
print(f"PS emulation: {emu.emulate_2d_ps}")
print("\nAvailable properties:")
print(f"  Redshifts: {emu.properties.zs.shape}")
print(f"  PS redshifts: {emu.properties.PS_zs.shape}")
print(f"  kperp: {emu.properties.kperp.shape}")
print(f"  kpar: {emu.properties.kpar.shape}")

## 3. Run Emulator Predictions

We run the emulator on a subset of test parameters.

In [ ]:
# Select a subset for demonstration (to save time)
n_samples = min(10, len(test_params))
idx = np.random.choice(len(test_params), n_samples, replace=False)
idx = np.sort(idx)

params_subset = test_params[idx]

print(f"Running emulator on {n_samples} samples...")

# Run prediction with EM sampling for 2D PS
normed_params, output, errors = emu.predict(
    params_subset, 
    verbose=True,
    ps_sampling_method='ode',  # probability flow ODE or Euler-Maruyama 'em' sampling supported
    n_realisations=10,  # Number of diffusion realisations
    
)

print(f"\nOutput keys: {list(output.keys())}")

In [ ]:
# Extract emulated outputs
emu_xHI = output['xHI']
emu_Tb = output['Tb']
emu_Ts = output['Ts']
emu_tau = output['tau']
emu_UVLFs = output['UVLFs']

# Extract true outputs for comparison
true_xHI = test_xHI[idx]
true_Tb = test_Tb[idx]
true_Ts = test_Ts[idx]
true_tau = test_tau[idx]
true_UVLFs = test_LFs[idx]

print(f"Emulated xHI shape: {emu_xHI.shape}")
print(f"Emulated Tb shape: {emu_Tb.shape}")
print(f"Emulated Ts shape: {emu_Ts.shape}")
print(f"Emulated tau shape: {emu_tau.shape}")
print(f"Emulated UVLFs shape: {emu_UVLFs.shape}")

## 4. Compare 1D Summaries: True vs Emulated

We plot several test samples comparing the true (simulation) values with emulated predictions.

In [ ]:
def plot_true_vs_emu(x, y_true, y_emu, x_label, y_label, 
                      xlims=None, N=10, offset=0,
                      cs=None, leg_loc=(0.5, 0.5), title=None):
    """Plot true vs emulated summaries with fractional error statistics."""
    if cs is None:
        cs = plt.cm.tab10(np.linspace(0, 1, N))
    
    y_diff = np.abs(y_true - y_emu)
    
    fig, axs = plt.subplots(
        nrows=2, ncols=1, sharex=True, figsize=(12, 8),
        gridspec_kw=dict(height_ratios=[3, 2], hspace=0)
    )
    axs = axs.flatten()

    diff_err_z = np.nanpercentile(y_diff, [2.5, 16, 50, 84, 97.5], axis=0)
    
    for i, c in zip(range(N), cs):
        if i == N - 1:
            labels = ['Emulated', 'True (simulation)']
        else:
            labels = [None, None]
        axs[0].plot(x, y_true[i + offset], lw=2, color=c, label=labels[1])
        axs[0].plot(x, y_emu[i + offset], lw=1.5, ls='--', color=c, label=labels[0])
        axs[1].plot(x, y_diff[i + offset], ls='-', alpha=0.5, lw=1, color=c)

    axs[0].legend(loc=leg_loc, frameon=False, fontsize=12)
    axs[1].plot(x, diff_err_z[2], ls='--', lw=2, color='k', label='Median')
    axs[1].fill_between(x, diff_err_z[1], diff_err_z[3], color='k', alpha=0.2, label='68% CI')
    axs[1].fill_between(x, diff_err_z[0], diff_err_z[4], color='k', alpha=0.1, label='95% CI')

    handles = [
        mpatches.Patch(color='k', label='68% CI', alpha=0.3),
        mpatches.Patch(color='k', label='95% CI', alpha=0.1),
    ]
    axs[1].legend(handles=handles, loc='upper right', frameon=False, fontsize=10)
    
    axs[0].set_ylabel(y_label, fontsize=14)
    axs[1].set_ylabel('|Difference|', fontsize=12)
    axs[1].set_xlabel(x_label, fontsize=14)
    
    if title:
        axs[0].set_title(title, fontsize=16)
    
    if xlims is not None:
        plt.xlim(xlims[0], xlims[1])
    else:
        plt.xlim(min(x), max(x))
    
    plt.tight_layout()
    return fig

In [ ]:
# Plot neutral fraction
fig = plot_true_vs_emu(
    redshifts, true_xHI, emu_xHI,
    r'Redshift $z$', r'$\overline{x}_{\mathrm{HI}}$',
    xlims=[5.8, 21], N=10, leg_loc=(0.6, 0.5),
    title='Neutral Fraction'
)
plt.show()

In [ ]:
# Plot brightness temperature
fig = plot_true_vs_emu(
    redshifts, true_Tb, emu_Tb,
    r'Redshift $z$', r'$\overline{T}_b$ [mK]',
    xlims=[5.8, 21], N=10, leg_loc=(0.55, 0.02),
    title='Global Brightness Temperature'
)
plt.show()

In [ ]:
# Plot spin temperature (in log scale for clarity)
fig = plot_true_vs_emu(
    redshifts, np.log10(true_Ts + 1e-3), np.log10(emu_Ts + 1e-3),
    r'Redshift $z$', r'$\log_{10} T_s$ [K]',
    xlims=[5.8, 21], N=10, leg_loc=(0.6, 0.6),
    title='Spin Temperature'
)
plt.show()

In [ ]:
# Plot optical depth (histogram comparison)
fig, ax = plt.subplots(figsize=(10, 6))

bins = np.linspace(
    min(true_tau.min(), emu_tau.min()),
    max(true_tau.max(), emu_tau.max()),
    30
)

ax.hist(true_tau, bins=bins, alpha=0.6, label='True', density=True)
ax.hist(emu_tau, bins=bins, alpha=0.6, label='Emulated', density=True)

# Add 1:1 comparison
ax2 = fig.add_axes([0.55, 0.55, 0.35, 0.35])
ax2.scatter(true_tau, emu_tau, alpha=0.5, s=10)
lims = [min(ax2.get_xlim()[0], ax2.get_ylim()[0]), 
        max(ax2.get_xlim()[1], ax2.get_ylim()[1])]
ax2.plot(lims, lims, 'k--', lw=1)
ax2.set_xlabel(r'True $\tau_e$', fontsize=10)
ax2.set_ylabel(r'Emulated $\tau_e$', fontsize=10)

ax.set_xlabel(r'$\tau_e$', fontsize=14)
ax.set_ylabel('Density', fontsize=14)
ax.set_title('Optical Depth to Reionization', fontsize=16)
ax.legend(fontsize=12)

plt.tight_layout()
plt.show()

### Compute Fractional Errors

In [ ]:
def fractional_error(true, pred, floor=1e-2):
    """Compute fractional error with floor to avoid division by small numbers."""
    denom = np.abs(true.copy())
    denom[denom < floor] = floor
    return np.abs((true - pred) / denom) * 100.0

# Compute FEs
fe_xHI = fractional_error(true_xHI, emu_xHI, floor=0.01)
fe_Tb = fractional_error(true_Tb, emu_Tb, floor=5.0)  # 5 mK floor for Tb
fe_Ts = fractional_error(np.log10(true_Ts + 1e-3), np.log10(emu_Ts + 1e-3), floor=0.01)
fe_tau = fractional_error(true_tau, emu_tau, floor=0.01)

print("Fractional Error Statistics (%)")
print("=" * 50)
for name, fe in [('xHI', fe_xHI), ('Tb', fe_Tb), ('Ts', fe_Ts), ('tau', fe_tau)]:
    finite = fe[np.isfinite(fe)]
    print(f"{name:8s}: median={np.median(finite):6.2f}%  "
          f"68th={np.percentile(finite, 68):6.2f}%  "
          f"95th={np.percentile(finite, 95):6.2f}%")

## 5. 2D Power Spectrum Emulation

The v3 emulator uses a **score-based diffusion model** to emulate the 2D power spectrum. This allows:
- Probabilistic predictions with multiple samples
- Variance and covariance estimation from the sample distribution
- Two sampling methods: **Euler-Maruyama (EM)** and **Probability-flow ODE**

In [ ]:
# Select a few parameters for 2D PS demonstration
n_ps_samples = 5
ps_test_idx = np.random.choice(len(ps_params), n_ps_samples, replace=False)

# Prepare parameters - we need to combine astro params with redshift
# The emulator expects 11 astro params; redshift is added internally
ps_test_params = ps_params[ps_test_idx]

print(f"Testing 2D PS on {n_ps_samples} parameter sets")
print(f"Parameter shape: {ps_test_params.shape}")
if ps_redshifts is not None:
    print(f"Using {len(ps_redshifts)} test set redshifts: {ps_redshifts}")

# Run with more samples for variance estimation
# Pass ps_redshifts to match the test set redshifts
_, ps_output, ps_errors = emu.predict(
    ps_test_params,
    verbose=True,
    ps_2d_redshifts=ps_redshifts,  # Use test set redshifts for comparison
    ps_sampling_method='ode',
    n_ps_batch=5,
    n_realisations=100  # More samples for better variance estimate
)

In [ ]:
# Access PS outputs
# PS_2D shape: (n_params, n_redshifts, 32, 64) - median of diffusion samples
ps_emu = ps_output['PS_2D']  # 2D PS median
print(f"Emulated 2D PS shape (median): {ps_emu.shape}")

# Get the actual redshifts used for 2D PS emulation
ps_emu_zs = output.PS_2D_redshifts if hasattr(output, 'PS_2D_redshifts') else ps_redshifts
print(f"2D PS redshifts: {ps_emu_zs}")

# Also access 1D PS from LSTM
ps_1d = ps_output['PS']
print(f"1D PS shape: {ps_1d.shape}")

In [ ]:
# Plot 2D PS comparison for one parameter set at one redshift
param_idx = 0
z_idx = 5  # Middle redshift in our subset (0 to n_z_subset-1)

# Get k-grids from emulator properties
kperp_emu = emu.properties.kperp
kpar_emu = emu.properties.kpar

if HAS_PS2D_DB and ps_2d_means is not None:
    # Get true PS from database (already subsetted to match ps_redshifts)
    true_ps_2d_sel = ps_2d_means[ps_test_idx]  # Shape: (n_ps_samples, n_z_subset, 32, 64)
    emu_ps_2d_sel = ps_emu  # Shape: (n_ps_samples, n_z_subset, 32, 64)
    
    print(f"True PS shape: {true_ps_2d_sel.shape}")
    print(f"Emulated PS shape: {emu_ps_2d_sel.shape}")
    print(f"Comparing at z = {ps_redshifts[z_idx]:.2f}")

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))

    # True PS at selected redshift
    true_ps = true_ps_2d_sel[param_idx, z_idx]
    emu_ps = emu_ps_2d_sel[param_idx, z_idx]

    vmin = np.percentile(np.log10(true_ps + 1e-10), 5)
    vmax = np.percentile(np.log10(true_ps + 1e-10), 95)

    # Row 1: PS values
    im0 = axes[0, 0].pcolormesh(kperp_emu, kpar_emu, np.log10(true_ps).T, 
                                 vmin=vmin, vmax=vmax, cmap='inferno')
    axes[0, 0].set_title('True (simulation)', fontsize=14)
    axes[0, 0].set_xlabel(r'$k_\perp$ [Mpc$^{-1}$]')
    axes[0, 0].set_ylabel(r'$k_\parallel$ [Mpc$^{-1}$]')
    axes[0, 0].set_xscale('log')
    plt.colorbar(im0, ax=axes[0, 0], label=r'$\log_{10}$ PS [mK$^2$]')

    im1 = axes[0, 1].pcolormesh(kperp_emu, kpar_emu, np.log10(emu_ps).T,
                                 vmin=vmin, vmax=vmax, cmap='inferno')
    axes[0, 1].set_title('Emulated (diffusion)', fontsize=14)
    axes[0, 1].set_xlabel(r'$k_\perp$ [Mpc$^{-1}$]')
    axes[0, 1].set_ylabel(r'$k_\parallel$ [Mpc$^{-1}$]')
    axes[0, 1].set_xscale('log')
    plt.colorbar(im1, ax=axes[0, 1], label=r'$\log_{10}$ PS [mK$^2$]')

    # Difference
    diff = np.log10(emu_ps) - np.log10(true_ps)
    vlim = np.max(np.abs(diff))
    im2 = axes[0, 2].pcolormesh(kperp_emu, kpar_emu, diff.T,
                                 vmin=-vlim, vmax=vlim, cmap='RdBu_r')
    axes[0, 2].set_title('Difference (log)', fontsize=14)
    axes[0, 2].set_xlabel(r'$k_\perp$ [Mpc$^{-1}$]')
    axes[0, 2].set_ylabel(r'$k_\parallel$ [Mpc$^{-1}$]')
    axes[0, 2].set_xscale('log')
    plt.colorbar(im2, ax=axes[0, 2], label=r'$\Delta \log_{10}$ PS')

    # Row 2: FE and 1D slices
    fe_ps = fractional_error(true_ps, emu_ps, floor=0.01)

    im3 = axes[1, 0].pcolormesh(kperp_emu, kpar_emu, fe_ps.T,
                                 vmin=0, vmax=min(100, np.percentile(fe_ps, 95)),
                                 cmap='hot_r')
    axes[1, 0].set_title(f'Fractional Error (mean: {np.mean(fe_ps):.1f}%)', fontsize=14)
    axes[1, 0].set_xlabel(r'$k_\perp$ [Mpc$^{-1}$]')
    axes[1, 0].set_ylabel(r'$k_\parallel$ [Mpc$^{-1}$]')
    axes[1, 0].set_xscale('log')
    plt.colorbar(im3, ax=axes[1, 0], label='FE [%]')

    # 1D slice along kperp (fixed kpar)
    kpar_slice_idx = 16  # Middle kpar
    axes[1, 1].plot(kperp_emu, true_ps[:, kpar_slice_idx], 'k-', lw=2, label='True')
    axes[1, 1].plot(kperp_emu, emu_ps[:, kpar_slice_idx], 'r--', lw=2, label='Emulated')
    axes[1, 1].set_xscale('log')
    axes[1, 1].set_yscale('log')
    axes[1, 1].set_xlabel(r'$k_\perp$ [Mpc$^{-1}$]')
    axes[1, 1].set_ylabel(r'PS [mK$^2$]')
    axes[1, 1].set_title(f'1D slice at $k_\\parallel$ = {kpar_emu[kpar_slice_idx]:.3f} Mpc$^{{-1}}$', fontsize=14)
    axes[1, 1].legend()

    # FE histogram
    axes[1, 2].hist(fe_ps.ravel(), bins=50, alpha=0.7, density=True)
    axes[1, 2].axvline(np.mean(fe_ps), color='r', ls='--', lw=2, 
                       label=f'Mean: {np.mean(fe_ps):.1f}%')
    axes[1, 2].axvline(np.median(fe_ps), color='b', ls='--', lw=2,
                       label=f'Median: {np.median(fe_ps):.1f}%')
    axes[1, 2].set_xlabel('Fractional Error [%]')
    axes[1, 2].set_ylabel('Density')
    axes[1, 2].set_title('FE Distribution', fontsize=14)
    axes[1, 2].legend()

    plt.suptitle(f'2D Power Spectrum at z = {ps_redshifts[z_idx]:.1f}', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

else:
    # Without database, just show the emulated PS
    kperp_emu = emu.properties.kperp
    kpar_emu = emu.properties.kpar
    emu_ps = ps_emu[param_idx, z_idx]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    vmin = np.percentile(np.log10(emu_ps + 1e-10), 5)
    vmax = np.percentile(np.log10(emu_ps + 1e-10), 95)
    
    im0 = axes[0].pcolormesh(kperp_emu, kpar_emu, np.log10(emu_ps).T, 
                              vmin=vmin, vmax=vmax, cmap='inferno')
    axes[0].set_title('Emulated 2D PS', fontsize=14)
    axes[0].set_xlabel(r'$k_\perp$ [Mpc$^{-1}$]')
    axes[0].set_ylabel(r'$k_\parallel$ [Mpc$^{-1}$]')
    axes[0].set_xscale('log')
    plt.colorbar(im0, ax=axes[0], label=r'$\log_{10}$ PS [mK$^2$]')
    
    # 1D slice along kperp (fixed kpar)
    kpar_slice_idx = 16  # Middle kpar
    axes[1].plot(kperp_emu, emu_ps[:, kpar_slice_idx], 'r-', lw=2, label='Emulated')
    axes[1].set_xscale('log')
    axes[1].set_yscale('log')
    axes[1].set_xlabel(r'$k_\perp$ [Mpc$^{-1}$]')
    axes[1].set_ylabel(r'PS [mK$^2$]')
    axes[1].set_title(f'1D slice at $k_\\parallel$ = {kpar_emu[kpar_slice_idx]:.3f} Mpc$^{{-1}}$', fontsize=14)
    axes[1].legend()
    
    # 1D slice along kpar (fixed kperp)
    kperp_slice_idx = 16  # Middle kperp
    axes[2].plot(kpar_emu, emu_ps[kperp_slice_idx, :], 'r-', lw=2, label='Emulated')
    axes[2].set_yscale('log')
    axes[2].set_xlabel(r'$k_\parallel$ [Mpc$^{-1}$]')
    axes[2].set_ylabel(r'PS [mK$^2$]')
    axes[2].set_title(f'1D slice at $k_\\perp$ = {kperp_emu[kperp_slice_idx]:.3f} Mpc$^{{-1}}$', fontsize=14)
    axes[2].legend()
    
    z_val = ps_redshifts[z_idx] if ps_redshifts is not None else emu.properties.PS_zs[z_idx]
    plt.suptitle(f'Emulated 2D Power Spectrum at z = {z_val:.1f}', fontsize=16)
    plt.tight_layout()
    plt.show()
    
    print("\nNote: 2D PS test database not available - showing emulated PS only.")

## 6. Uncertainty Quantification from Diffusion Samples

The diffusion model generates multiple samples for each input, allowing variance estimation.

In [ ]:
# Access PS samples to compute variance
# Rerun with raw output to access samples
_, ps_output_raw, _ = emu.predict(
    ps_test_params[:1],  # Single parameter set
    verbose=False,
    ps_2d_redshifts=ps_redshifts,  # Use same redshifts as before
    ps_sampling_method='em',
    n_realisations=100
)

# Get the raw output which contains PS_2D_samples
ps_samples = ps_output_raw.PS_2D_samples  # Shape: (1, n_z, n_samples, 32, 64)
print(f"PS samples shape: {ps_samples.shape}")

# Compute variance per pixel
ps_mean = np.mean(ps_samples, axis=2)
ps_std = np.std(ps_samples, axis=2)
ps_var = np.var(ps_samples, axis=2)

print(f"PS mean shape: {ps_mean.shape}")
print(f"PS std shape: {ps_std.shape}")

In [ ]:
# Plot variance map for one redshift
z_idx = 5  # Use the same z_idx as the comparison plots

# Use emulator k-grids
kperp_emu = emu.properties.kperp
kpar_emu = emu.properties.kpar

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Mean PS
im0 = axes[0].pcolormesh(kperp_emu, kpar_emu, np.log10(ps_mean[0, z_idx]).T, cmap='inferno')
axes[0].set_title('Mean PS', fontsize=14)
axes[0].set_xlabel(r'$k_\perp$ [Mpc$^{-1}$]')
axes[0].set_ylabel(r'$k_\parallel$ [Mpc$^{-1}$]')
axes[0].set_xscale('log')
plt.colorbar(im0, ax=axes[0], label=r'$\log_{10}$ PS')

# Standard deviation
im1 = axes[1].pcolormesh(kperp_emu, kpar_emu, np.log10(ps_std[0, z_idx] + 1e-10).T, cmap='viridis')
axes[1].set_title('Standard Deviation', fontsize=14)
axes[1].set_xlabel(r'$k_\perp$ [Mpc$^{-1}$]')
axes[1].set_ylabel(r'$k_\parallel$ [Mpc$^{-1}$]')
axes[1].set_xscale('log')
plt.colorbar(im1, ax=axes[1], label=r'$\log_{10}$ $\sigma$')

# Coefficient of variation (std/mean)
cv = ps_std[0, z_idx] / (ps_mean[0, z_idx] + 1e-10)
im2 = axes[2].pcolormesh(kperp_emu, kpar_emu, cv.T, cmap='hot_r', vmin=0, vmax=1)
axes[2].set_title('Coefficient of Variation', fontsize=14)
axes[2].set_xlabel(r'$k_\perp$ [Mpc$^{-1}$]')
axes[2].set_ylabel(r'$k_\parallel$ [Mpc$^{-1}$]')
axes[2].set_xscale('log')
plt.colorbar(im2, ax=axes[2], label='CV = σ/μ')

z_label = ps_redshifts[z_idx] if ps_redshifts is not None else z_idx
plt.suptitle(f'Variance from Diffusion Samples (z = {z_label:.1f})', fontsize=16)
plt.tight_layout()
plt.show()

### Covariance and Correlation Analysis

The diffusion model samples allow us to compute the full covariance matrix between all $(k_\perp, k_\parallel)$ pixels. This reveals the correlations induced by the diffusion process.

In [ ]:
# Compute full covariance matrix from samples
# Flatten samples to (n_samples, n_pixels)
H, W = ps_samples.shape[-2], ps_samples.shape[-1]
n_samples_cov = ps_samples.shape[2]
n_pixels = H * W

samples_flat = ps_samples[0, z_idx].reshape(n_samples_cov, -1)  # (n_samples, H*W)

# Center the samples
samples_centered = samples_flat - samples_flat.mean(axis=0, keepdims=True)

# Compute covariance matrix: Cov[i,j] = E[(X_i - μ_i)(X_j - μ_j)]
cov_matrix = (samples_centered.T @ samples_centered) / (n_samples_cov - 1)
print(f"Covariance matrix shape: {cov_matrix.shape}")

# Compute correlation matrix from covariance
std_vec = np.sqrt(np.diag(cov_matrix))
std_outer = np.outer(std_vec, std_vec)
corr_matrix = cov_matrix / (std_outer + 1e-10)

# Clip correlation to [-1, 1] due to numerical precision
corr_matrix = np.clip(corr_matrix, -1, 1)
print(f"Correlation matrix shape: {corr_matrix.shape}")

# Compute diagnostic statistics
diag_var = np.diag(cov_matrix)
total_var = np.sum(diag_var)
off_diag_var = np.sum(np.abs(cov_matrix)) - total_var
diag_frac = total_var / (total_var + off_diag_var)

off_diag_mask = ~np.eye(n_pixels, dtype=bool)
mean_abs_corr = np.mean(np.abs(corr_matrix[off_diag_mask]))
max_abs_corr = np.max(np.abs(corr_matrix[off_diag_mask]))

z_label = ps_redshifts[z_idx] if ps_redshifts is not None else z_idx
print(f"\nCovariance diagnostics at z = {z_label:.1f}:")
print(f"  Diagonal fraction: {diag_frac:.3f}")
print(f"  Mean |off-diag correlation|: {mean_abs_corr:.3f}")
print(f"  Max |off-diag correlation|: {max_abs_corr:.3f}")

In [ ]:
# Plot correlation matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Full correlation matrix
im0 = axes[0].imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
axes[0].set_title(f'Pixel Correlation Matrix ({n_pixels}x{n_pixels})', fontsize=14)
axes[0].set_xlabel('Pixel index (flattened)')
axes[0].set_ylabel('Pixel index (flattened)')
plt.colorbar(im0, ax=axes[0], label='Correlation')

# Add grid lines to show 32x64 block structure
for i in range(0, n_pixels+1, W):
    axes[0].axhline(i-0.5, color='gray', alpha=0.3, lw=0.5)
    axes[0].axvline(i-0.5, color='gray', alpha=0.3, lw=0.5)

# Histogram of off-diagonal correlations
off_diag_corrs = corr_matrix[off_diag_mask]
axes[1].hist(off_diag_corrs, bins=100, alpha=0.7, density=True, color='steelblue')
axes[1].axvline(0, color='k', ls='--', lw=1)
axes[1].axvline(mean_abs_corr, color='r', ls='--', lw=2, 
                label=f'Mean |corr| = {mean_abs_corr:.3f}')
axes[1].axvline(-mean_abs_corr, color='r', ls='--', lw=2)
axes[1].set_xlabel('Correlation coefficient', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title('Distribution of Off-Diagonal Correlations', fontsize=14)
axes[1].legend(loc='upper right')
axes[1].set_xlim(-1, 1)

z_label = ps_redshifts[z_idx] if ps_redshifts is not None else z_idx
plt.suptitle(f'Correlation Structure (z = {z_label:.1f})', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Spatial correlation maps: correlation of each pixel with reference pixels
kperp_emu = emu.properties.kperp
kpar_emu = emu.properties.kpar

# Define reference pixels: center and corner
center_idx = (H // 2) * W + (W // 2)  # Center pixel
corner_idx = 0  # Top-left corner
mid_k_idx = (H // 4) * W + (W // 2)  # Mid-k region

# Extract correlation with reference pixels and reshape to 2D
corr_with_center = corr_matrix[center_idx, :].reshape(H, W)
corr_with_corner = corr_matrix[corner_idx, :].reshape(H, W)
corr_with_mid = corr_matrix[mid_k_idx, :].reshape(H, W)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Correlation with center pixel
im0 = axes[0].pcolormesh(kperp_emu, kpar_emu, corr_with_center.T, 
                          cmap='RdBu_r', vmin=-1, vmax=1)
axes[0].plot(kperp_emu[H//2], kpar_emu[W//2], 'k*', ms=15, label='Reference')
axes[0].set_title('Correlation with Center Pixel', fontsize=14)
axes[0].set_xlabel(r'$k_\perp$ [Mpc$^{-1}$]')
axes[0].set_ylabel(r'$k_\parallel$ [Mpc$^{-1}$]')
axes[0].set_xscale('log')
axes[0].legend()
plt.colorbar(im0, ax=axes[0], label='Correlation')

# Correlation with corner pixel (low-k)
im1 = axes[1].pcolormesh(kperp_emu, kpar_emu, corr_with_corner.T,
                          cmap='RdBu_r', vmin=-1, vmax=1)
axes[1].plot(kperp_emu[0], kpar_emu[0], 'k*', ms=15, label='Reference')
axes[1].set_title('Correlation with Low-k Corner', fontsize=14)
axes[1].set_xlabel(r'$k_\perp$ [Mpc$^{-1}$]')
axes[1].set_ylabel(r'$k_\parallel$ [Mpc$^{-1}$]')
axes[1].set_xscale('log')
axes[1].legend()
plt.colorbar(im1, ax=axes[1], label='Correlation')

# Correlation with mid-k pixel
im2 = axes[2].pcolormesh(kperp_emu, kpar_emu, corr_with_mid.T,
                          cmap='RdBu_r', vmin=-1, vmax=1)
axes[2].plot(kperp_emu[H//4], kpar_emu[W//2], 'k*', ms=15, label='Reference')
axes[2].set_title('Correlation with Mid-k Pixel', fontsize=14)
axes[2].set_xlabel(r'$k_\perp$ [Mpc$^{-1}$]')
axes[2].set_ylabel(r'$k_\parallel$ [Mpc$^{-1}$]')
axes[2].set_xscale('log')
axes[2].legend()
plt.colorbar(im2, ax=axes[2], label='Correlation')

z_label = ps_redshifts[z_idx] if ps_redshifts is not None else z_idx
plt.suptitle(f'Spatial Correlation Maps (z = {z_label:.1f})', fontsize=16)
plt.tight_layout()
plt.show()

print("\nCorrelation length analysis:")
print(f"  Center pixel self-correlation: {corr_with_center[H//2, W//2]:.3f}")
print(f"  Center-corner correlation: {corr_with_center[0, 0]:.3f}")
print(f"  This indicates {'strong' if abs(corr_with_center[0, 0]) > 0.3 else 'weak'} "
      f"long-range correlations in the diffusion samples.")

## 7. Comparison: EM vs ODE Sampling

The diffusion model supports two sampling methods:
- **Euler-Maruyama (EM)**: Stochastic sampling, faster, allows variance estimation
- **Probability-flow ODE**: Deterministic sampling, slower but exact likelihood

In [ ]:
import time

# Single parameter for comparison
test_single = ps_test_params[:1]

# EM sampling
t0 = time.time()
_, out_em, _ = emu.predict(test_single, ps_2d_redshifts=ps_redshifts, 
                           ps_sampling_method='em', n_realisations=50, verbose=False)
t_em = time.time() - t0

# ODE sampling
t0 = time.time()
_, out_ode, _ = emu.predict(test_single, ps_2d_redshifts=ps_redshifts,
                            ps_sampling_method='ode', n_realisations=50, verbose=False)
t_ode = time.time() - t0

print(f"EM sampling:  {t_em:.2f}s")
print(f"ODE sampling: {t_ode:.2f}s")

In [ ]:
# Compare EM vs ODE - use same z_idx as before
z_idx_compare = 5  # Should match the subset of redshifts we selected

ps_em = out_em['PS_2D'][0, z_idx_compare]
ps_ode = out_ode['PS_2D'][0, z_idx_compare]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

vmin = min(np.log10(ps_em).min(), np.log10(ps_ode).min())
vmax = max(np.log10(ps_em).max(), np.log10(ps_ode).max())

kperp_emu = emu.properties.kperp
kpar_emu = emu.properties.kpar

im0 = axes[0].pcolormesh(kperp_emu, kpar_emu, np.log10(ps_em).T, 
                          vmin=vmin, vmax=vmax, cmap='inferno')
axes[0].set_title(f'EM Sampling ({t_em:.1f}s)', fontsize=14)
axes[0].set_xlabel(r'$k_\perp$ [Mpc$^{-1}$]')
axes[0].set_ylabel(r'$k_\parallel$ [Mpc$^{-1}$]')
axes[0].set_xscale('log')
plt.colorbar(im0, ax=axes[0], label=r'$\log_{10}$ PS')

im1 = axes[1].pcolormesh(kperp_emu, kpar_emu, np.log10(ps_ode).T,
                          vmin=vmin, vmax=vmax, cmap='inferno')
axes[1].set_title(f'ODE Sampling ({t_ode:.1f}s)', fontsize=14)
axes[1].set_xlabel(r'$k_\perp$ [Mpc$^{-1}$]')
axes[1].set_ylabel(r'$k_\parallel$ [Mpc$^{-1}$]')
axes[1].set_xscale('log')
plt.colorbar(im1, ax=axes[1], label=r'$\log_{10}$ PS')

# Difference
diff = np.log10(ps_em) - np.log10(ps_ode)
vlim = np.percentile(np.abs(diff), 95)
im2 = axes[2].pcolormesh(kperp_emu, kpar_emu, diff.T,
                          vmin=-vlim, vmax=vlim, cmap='RdBu_r')
axes[2].set_title(f'EM - ODE (RMS: {np.sqrt(np.mean(diff**2)):.3f})', fontsize=14)
axes[2].set_xlabel(r'$k_\perp$ [Mpc$^{-1}$]')
axes[2].set_ylabel(r'$k_\parallel$ [Mpc$^{-1}$]')
axes[2].set_xscale('log')
plt.colorbar(im2, ax=axes[2], label=r'$\Delta \log_{10}$ PS')

z_label = ps_redshifts[z_idx_compare] if ps_redshifts is not None else z_idx_compare
plt.suptitle(f'EM vs ODE Sampling Comparison (z = {z_label:.1f})', fontsize=16)
plt.tight_layout()
plt.show()

## Using the MHEmulatorErrors Class

The MH (v3) emulator provides error statistics via the `MHEmulatorErrors` class, which differs significantly from ACG/Radio emulators:

**Key Differences:**
- **Absolute errors** in physical units (not fractional %)
- **Output-dependent** errors: computed from `FE% × |output_value|`
- **2D PS error statistics**: variance, covariance, and correlation matrices
- **Sampling method support**: different errors for 'em' vs 'ode' sampling

In [ ]:
# Get the error object from the prediction
_, output_single, errors = emu.predict(test_params[0:1], verbose=False)

# Inspect the error object
print(f"Error type: {type(errors).__name__}")
print(f"\nAvailable error fields: {errors.keys()}")
print(f"\nError summary:\n{errors.summary()}")

### Absolute Errors with Physical Units

Unlike ACG/Radio which store FE%, MHEmulatorErrors provides **absolute errors** with proper astropy units. For log-space quantities (PS, UVLFs), errors are in **dex** (log10 units).

In [ ]:
# Check units of each error field
print("Error field units:")
print(f"  PS_err: {errors.PS_err.unit} (dex - error on log10 PS)")
print(f"  Tb_err: {errors.Tb_err.unit} (absolute mK error)")
print(f"  xHI_err: {errors.xHI_err.unit} (absolute neutral fraction error)")
print(f"  Ts_err: {errors.Ts_err.unit} (absolute K error)")
print(f"  tau_err: {errors.tau_err.unit} (absolute optical depth error)")
print(f"  UVLFs_logerr: {errors.UVLFs_logerr.unit} (dex - error on log10 LF)")

# Median PS error interpretation
median_ps_err = np.nanmedian(errors.PS_err.value)
print(f"\nMedian PS error: {median_ps_err:.3f} dex")
print(f"  This means log10(PS) predictions are typically off by ~{median_ps_err:.3f}")
print(f"  In linear PS, this is a factor of 10^{median_ps_err:.3f} ≈ {10**median_ps_err:.2f} ({(10**median_ps_err - 1)*100:.0f}% error)")

### 2D Power Spectrum Error Statistics (Unique to MH Emulator)

The MH emulator's diffusion model provides advanced error statistics for the 2D power spectrum:

- **Variance**: Per-bin error variance from test set residuals
- **Covariance**: Full covariance matrix between (kperp, kpar) bins
- **Correlation diagnostics**: `ps_diagonal_fraction`, `ps_mean_abs_correlation`

In [ ]:
# Access 2D PS error statistics via the errors object
ps_var = errors.get_ps_variance()
ps_cov = errors.get_ps_covariance()

if ps_var is not None:
    print(f"2D PS variance shape: {ps_var.shape}")
    print(f"2D PS covariance shape: {ps_cov.shape if ps_cov is not None else 'N/A'}")
    print("\nCorrelation diagnostics:")
    print(f"  Diagonal fraction: {errors.ps_diagonal_fraction:.2%}")
    print("  (1.0 = uncorrelated errors, <1.0 = some correlation)")
    print(f"  Mean |off-diag correlation|: {errors.ps_mean_abs_correlation:.3f}")
    print("  (0 = uncorrelated, 1 = perfectly correlated)")
else:
    print("2D PS statistics not available (emulator loaded without 2D PS)")

In [ ]:
# Visualize 2D PS variance and correlation matrix
if ps_var is not None and ps_cov is not None:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Get k-grids from emulator properties
    kperp = emu.properties.kperp
    kpar = emu.properties.kpar
    
    # Plot variance map
    im0 = axes[0].pcolormesh(kperp, kpar, ps_var.T, cmap='viridis')
    axes[0].set_xlabel(r'$k_\perp$ [Mpc$^{-1}$]')
    axes[0].set_ylabel(r'$k_\parallel$ [Mpc$^{-1}$]')
    axes[0].set_xscale('log')
    axes[0].set_title('2D PS Error Variance (FE%²)')
    plt.colorbar(im0, ax=axes[0], label='Variance')
    
    # Plot covariance matrix (subsampled for visibility)
    step = 8  # Subsample for clearer visualization
    cov_sub = ps_cov[::step, ::step]
    im1 = axes[1].imshow(cov_sub, cmap='RdBu_r', aspect='equal',
                         vmin=-np.percentile(np.abs(cov_sub), 95),
                         vmax=np.percentile(np.abs(cov_sub), 95))
    axes[1].set_title(f'Covariance Matrix (subsampled {step}x)')
    axes[1].set_xlabel('Pixel index')
    axes[1].set_ylabel('Pixel index')
    plt.colorbar(im1, ax=axes[1], label='Covariance')
    
    # Compute and plot correlation matrix
    std = np.sqrt(np.diag(ps_cov))
    std_outer = np.outer(std, std)
    corr = ps_cov / std_outer
    corr_sub = corr[::step, ::step]
    
    im2 = axes[2].imshow(corr_sub, cmap='RdBu_r', aspect='equal', vmin=-1, vmax=1)
    axes[2].set_title('Correlation Matrix (subsampled)')
    axes[2].set_xlabel('Pixel index')
    axes[2].set_ylabel('Pixel index')
    plt.colorbar(im2, ax=axes[2], label='Correlation')
    
    plt.suptitle('MH Emulator: 2D PS Error Statistics', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("Load emulator with emulate_2d_ps=True to access these statistics")

## Summary

This tutorial demonstrated the v3 (MH) emulator capabilities:

1. **1D Summaries**: Global $T_b$, $x_{\mathrm{HI}}$, $T_s$, UVLFs, and $\tau_e$ emulated by LSTM networks
2. **2D Power Spectrum**: $P(k_\perp, k_\parallel)$ emulated by a score-based diffusion model
3. **Uncertainty Quantification**: 
   - Variance/standard deviation maps from diffusion samples
   - Full covariance matrix computation between PS pixels
   - Correlation structure analysis and spatial correlation maps
4. **Sampling Methods**: EM (fast, stochastic) vs ODE (slower, deterministic)

The emulator achieves:
- Sub-percent median fractional errors on most 1D summaries
- ~10% median FE on 2D PS for most k-modes
- Full probabilistic uncertainty through diffusion sampling
- Pixel-by-pixel covariance for downstream statistical analyses